In [ ]:
!pip install scikit-learn xgboost lightgbm -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import randint, uniform
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## 1. Linear Regression (Ridge) — GridSearchCV

In [ ]:
param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'fit_intercept': [True, False],
    'solver': ['auto', 'svd', 'cholesky', 'lsqr']
}

search = GridSearchCV(Ridge(), param_grid, cv=5, scoring='accuracy')
search.fit(X_train_scaled, y_train)

print("Best Hyperparameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

## 2. Decision Tree — GridSearchCV

In [ ]:
param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None, 3, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2']
}

search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
search.fit(X_train, y_train)

print("Best Hyperparameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

## 3. Random Forest — RandomizedSearchCV

In [ ]:
param_dist = {
    'n_estimators': randint(50, 500),
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None],
    'criterion': ['gini', 'entropy']
}

search = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_dist,
                            n_iter=30, cv=5, scoring='accuracy', random_state=42)
search.fit(X_train, y_train)

print("Best Hyperparameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

## 4. Support Vector Machine (SVM) — GridSearchCV

In [ ]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3, 4]
}

search = GridSearchCV(SVC(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
search.fit(X_train_scaled, y_train)

print("Best Hyperparameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

## 5. Neural Network (MLP) — RandomizedSearchCV

In [ ]:
param_dist = {
    'hidden_layer_sizes': [(50,), (100,), (100, 50), (128, 64), (64, 64, 32)],
    'activation': ['relu', 'tanh', 'logistic'],
    'solver': ['adam', 'sgd', 'lbfgs'],
    'alpha': uniform(0.0001, 0.1),
    'learning_rate': ['constant', 'adaptive', 'invscaling'],
    'max_iter': [200, 500, 1000]
}

search = RandomizedSearchCV(MLPClassifier(random_state=42), param_dist,
                            n_iter=25, cv=5, scoring='accuracy', random_state=42)
search.fit(X_train_scaled, y_train)

print("Best Hyperparameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

## 6. K-Nearest Neighbors (KNN) — GridSearchCV

In [ ]:
param_grid = {
    'n_neighbors': list(range(1, 21)),
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'p': [1, 2]
}

search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
search.fit(X_train_scaled, y_train)

print("Best Hyperparameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

## 7a. Gradient Boosting — RandomizedSearchCV

In [ ]:
param_dist = {
    'n_estimators': randint(50, 400),
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': randint(2, 8),
    'subsample': uniform(0.5, 0.5),
    'min_samples_split': randint(2, 20),
    'max_features': ['sqrt', 'log2', None]
}

search = RandomizedSearchCV(GradientBoostingClassifier(random_state=42), param_dist,
                            n_iter=25, cv=5, scoring='accuracy', random_state=42)
search.fit(X_train, y_train)

print("Best Hyperparameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

## 7b. XGBoost — RandomizedSearchCV

In [ ]:
param_dist = {
    'n_estimators': randint(50, 400),
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': randint(2, 10),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0.5, 2),
    'gamma': uniform(0, 0.5)
}

search = RandomizedSearchCV(XGBClassifier(verbosity=0, eval_metric='logloss', random_state=42),
                            param_dist, n_iter=25, cv=5, scoring='accuracy', random_state=42)
search.fit(X_train, y_train)

print("Best Hyperparameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

## 7c. LightGBM — RandomizedSearchCV

In [ ]:
param_dist = {
    'n_estimators': randint(50, 400),
    'learning_rate': uniform(0.01, 0.3),
    'num_leaves': randint(20, 150),
    'max_depth': [-1, 5, 10, 20],
    'min_child_samples': randint(5, 50),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 1)
}

search = RandomizedSearchCV(LGBMClassifier(random_state=42, verbose=-1),
                            param_dist, n_iter=25, cv=5, scoring='accuracy', random_state=42)
search.fit(X_train, y_train)

print("Best Hyperparameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

## 8. K-Means — Silhouette Score Search
K-Means is unsupervised, so GridSearchCV/RandomizedSearchCV don't apply. We loop over hyperparameters and pick the config with the best silhouette score.

In [ ]:
X_scaled = StandardScaler().fit_transform(X)

best_score = -1
best_params = {}

for k in range(2, 11):
    for init in ['k-means++', 'random']:
        for algo in ['lloyd', 'elkan']:
            km = KMeans(n_clusters=k, init=init, algorithm=algo, n_init=10, random_state=42)
            labels = km.fit_predict(X_scaled)
            score = silhouette_score(X_scaled, labels)
            if score > best_score:
                best_score = score
                best_params = {'n_clusters': k, 'init': init, 'algorithm': algo}

print("Best Hyperparameters:", best_params)
print("Best Silhouette Score:", round(best_score, 4))